# FC-PINO model prediction pointwise residual visualization for `elasticity`

This notebook visualizes the FC-PINO pointwise residual from the trained two-stage data+physics checkpoint.

The plotted residuals use the checkpoint prediction and the derivatives returned by `FC_FNO(model_input, derivs_to_compute=['dx', 'dy'])`. No CG1/FEM residual reconstruction is used for plotting. `r1` is the constitutive-energy density and `r2` is the equilibrium density, matching `ElasticityLeastSquares.compute_physical_loss_{1,2}`.

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import scipy.io
import torch
import torch.nn.functional as F
from tqdm import tqdm

import dolfinx
import scifem
import ufl

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

notebook_cwd = Path.cwd().resolve()
# Launched from FC_PINO/ or from the repo root; derive the repo root either way.
repo_path = notebook_cwd.parent if notebook_cwd.name == 'FC_PINO' else notebook_cwd
fc_pino_path = repo_path / 'FC_PINO'
for path in [str(repo_path), str(fc_pino_path)]:
    while path in sys.path:
        sys.path.remove(path)
for path in [str(fc_pino_path), str(repo_path)]:
    sys.path.insert(0, path)

from data_generation.differential_equations import ElasticityLeastSquares
from utils import evaluate_expression, load_yaml
from fc_fno import FC_FNO
from neuralop.layers.fourier_continuation import FCLegendre, FCGram

torch.set_default_dtype(torch.float64)
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f'repo_path: {repo_path}')
print(f'device: {device}')

## Configuration

Defaults visualize one test sample from the requested checkpoint and save figures under `results/elasticity/model_test_outputs/fc_pino_two_stage_data_physics/visualizations`.

In [ ]:
mesh_config_path = repo_path / 'configs/elasticity/config_data/config_mesh.yaml'
function_space_config_path = repo_path / 'configs/elasticity/config_data/config_function_space.yaml'
train_dataset_path = repo_path / 'results/elasticity/train_dataset'
test_dataset_path = repo_path / 'results/elasticity/test_dataset'
model_train_outputs_path = repo_path / 'results/elasticity/model_train_outputs/fc_pino_two_stage_data_physics'
model_test_outputs_path = repo_path / 'results/elasticity/model_test_outputs/fc_pino_two_stage_data_physics'
visualization_dir = model_test_outputs_path / 'visualizations'
visualization_dir.mkdir(parents=True, exist_ok=True)

checkpoint_path = model_train_outputs_path / 'best_physics_model_params_N1_2000_N2_3000_coords.pth'
assert checkpoint_path.exists(), checkpoint_path

# This checkpoint was produced by elasticity_fc_pino_two_stage_data_physics.py.
dataset_split = 'test'  # 'train' or 'test'
sample_start_index = 1
num_check = 1
batch_size = 1

fc_backend = 'legendre'  # 'gram' or 'legendre'
fc_degree = 1
fc_cont_points = 50
n_modes = (32, 32)
hidden_channels = 128
n_layers = 4

mesh_args = load_yaml(mesh_config_path)
function_space_args = load_yaml(function_space_config_path)
num_x = mesh_args['num_x']
num_y = mesh_args['num_y']
Lx = mesh_args['upper_right_x'] - mesh_args['lower_left_x']
Ly = mesh_args['upper_right_y'] - mesh_args['lower_left_y']
print(mesh_args)
print(f'checkpoint: {checkpoint_path}')
print(f'visualization_dir: {visualization_dir}')

## Data loading

Tensors use `(batch, channel, x_index, y_index)`, matching the FC-PINO training notebooks. The model input is `[p, x, y]` and the output channels are `[sigma_11, sigma_12, sigma_21, sigma_22, u_1, u_2]`.

In [ ]:
dataset_path = train_dataset_path if dataset_split == 'train' else test_dataset_path


def load_slice(path: Path, start: int, count: int):
    arr = np.load(path, mmap_mode='r')
    if start < 0 or start >= arr.shape[0]:
        raise IndexError(f'sample_start_index={start} is outside {path.name} with {arr.shape[0]} samples')
    stop = min(start + count, arr.shape[0])
    return np.array(arr[start:stop])


def scalar_vertices_to_xy(values):
    return values.reshape(values.shape[0], num_x + 1, num_y + 1)


def tensor_vertices_to_cxy(values):
    num_components = values.shape[2]
    return values.reshape(values.shape[0], num_x + 1, num_y + 1, num_components).transpose(0, 3, 1, 2)

p_vertex = load_slice(dataset_path / 'p_vertex_values.npy', sample_start_index, num_check)
sigma_vertex = load_slice(dataset_path / 'sigma_vertex_values.npy', sample_start_index, num_check)
u_vertex = load_slice(dataset_path / 'u_vertex_values.npy', sample_start_index, num_check)
p_dof = load_slice(dataset_path / 'p_dof.npy', sample_start_index, num_check)
sigma_u_dof = load_slice(dataset_path / 'sigma_u_dof.npy', sample_start_index, num_check)
num_check = p_vertex.shape[0]

p_xy = scalar_vertices_to_xy(p_vertex)[:, None, :, :]
reference_sigma_u_xy = np.concatenate([
    tensor_vertices_to_cxy(sigma_vertex),
    tensor_vertices_to_cxy(u_vertex),
], axis=1)

p_tensor = torch.as_tensor(p_xy, dtype=torch.float64)
reference_sigma_u_tensor = torch.as_tensor(reference_sigma_u_xy, dtype=torch.float64)

class AddSpatialCoordinates(torch.nn.Module):
    def __init__(self, num_x, num_y, x_min=0.0, x_max=1.0, y_min=0.0, y_max=1.0):
        super().__init__()
        x = torch.linspace(x_min, x_max, num_x, dtype=torch.float64)
        y = torch.linspace(y_min, y_max, num_y, dtype=torch.float64)
        x_coor, y_coor = torch.meshgrid(x, y, indexing='ij')
        self.register_buffer('x_coor', x_coor[None, None, :, :])
        self.register_buffer('y_coor', y_coor[None, None, :, :])

    def forward(self, inputs):
        batch_size = inputs.shape[0]
        x = self.x_coor.expand(batch_size, -1, -1, -1).to(device=inputs.device, dtype=inputs.dtype)
        y = self.y_coor.expand(batch_size, -1, -1, -1).to(device=inputs.device, dtype=inputs.dtype)
        return torch.cat((inputs, x, y), dim=1)

add_spatial_coordinates = AddSpatialCoordinates(
    num_x + 1,
    num_y + 1,
    x_min=mesh_args['lower_left_x'],
    x_max=mesh_args['upper_right_x'],
    y_min=mesh_args['lower_left_y'],
    y_max=mesh_args['upper_right_y'],
)
model_input_tensor = add_spatial_coordinates(p_tensor)
print('p:', tuple(p_tensor.shape), 'model input:', tuple(model_input_tensor.shape), 'reference sigma_u:', tuple(reference_sigma_u_tensor.shape))

## Elasticity auxiliary fields

This reproduces the `q` (`z = grad(q)`) and `w` auxiliary fields used by the least-squares residual. Both are independent of `p`. The Lame parameters use `E = 1 + exp(p)`, `nu = 0.4`.

In [ ]:
elasticity_least_squares = ElasticityLeastSquares(mesh_args, function_space_args)
mesh = elasticity_least_squares.mesh
Vh = elasticity_least_squares.Vh

dolfinx_mesh_coords = mesh.geometry.x[:, :2]
x_grid = np.linspace(mesh_args['lower_left_x'], mesh_args['upper_right_x'], num_x + 1)
y_grid = np.linspace(mesh_args['lower_left_y'], mesh_args['upper_right_y'], num_y + 1)
image_mesh_coords = np.array(np.meshgrid(x_grid, y_grid)).T.reshape(-1, 2)
perm = np.array([np.where(np.isclose(image_mesh_coords, row).all(axis=1))[0][0] for row in dolfinx_mesh_coords], dtype=np.int64)
max_perm_coordinate_mismatch = np.max(np.abs(image_mesh_coords[perm] - dolfinx_mesh_coords))
print(f'max mesh/image coordinate mismatch after perm: {max_perm_coordinate_mismatch:.2e}')
assert max_perm_coordinate_mismatch < 1e-12

q_fc = elasticity_least_squares.solve_q()
w_fc = elasticity_least_squares.solve_w()
q_grad_mesh = evaluate_expression(mesh, ufl.grad(q_fc), mesh.geometry.x)[1]
w_grad_mesh = evaluate_expression(mesh, ufl.grad(w_fc), mesh.geometry.x)[1]


def mesh_values_to_xy(values_at_mesh):
    values_at_mesh = np.asarray(values_at_mesh)
    if values_at_mesh.ndim == 1:
        values_at_mesh = values_at_mesh[:, None]
    flat = np.zeros((len(perm), values_at_mesh.shape[1]), dtype=np.float64)
    flat[perm, :] = values_at_mesh
    return flat.reshape(num_x + 1, num_y + 1, values_at_mesh.shape[1]).transpose(2, 0, 1)

z_grad_xy = mesh_values_to_xy(q_grad_mesh)
w_grad_xy = mesh_values_to_xy(w_grad_mesh)

z_grad_tensor = torch.as_tensor(z_grad_xy[None, :, :, :], dtype=torch.float64, device=device)
w_grad_tensor = torch.as_tensor(w_grad_xy[None, :, :, :], dtype=torch.float64, device=device)

youngs_modulus_base = 1.0
poisson_ratio = 0.4


def lame_from_p(p):
    youngs_modulus = youngs_modulus_base + torch.exp(p)
    lambda_ = (youngs_modulus * poisson_ratio) / ((1.0 + poisson_ratio) * (1.0 - 2.0 * poisson_ratio))
    mu = youngs_modulus / (2.0 * (1.0 + poisson_ratio))
    return lambda_, mu

print('z=grad(q):', tuple(z_grad_tensor.shape), 'grad(w):', tuple(w_grad_tensor.shape))

## Model inference and FC-grid residual

The FC residual here uses the derivatives returned by `FC_FNO(model_input, derivs_to_compute=['dx', 'dy'])`, i.e. the same derivative path used during physics training.

In [ ]:
def ensure_fcgram_npz_matrices(d_values=(2, 3, 4, 5, 6), c=25):
    src_dir = fc_pino_path / 'FC_Gram_Construction/FCGram_matrices'
    dst_dir = model_test_outputs_path / 'fcgram_matrices'
    dst_dir.mkdir(parents=True, exist_ok=True)
    for d in d_values:
        dst = dst_dir / f'FCGram_data_d{d}_c{c}.npz'
        if dst.exists():
            continue
        src = src_dir / f'FCGram_data_d{d}_C{c}.mat'
        mat = scipy.io.loadmat(src)
        np.savez(dst, ArQr=mat['ArQr'], AlQl=mat['AlQl'])
    return dst_dir

if fc_backend.lower() == 'gram':
    if FCGram is None:
        raise ImportError("FCGram is not available in this environment; set fc_backend = 'legendre' or install FCGram support.")
    fcgram_matrices_path = ensure_fcgram_npz_matrices(d_values=(fc_degree,), c=fc_cont_points // 2)
    extension_func = FCGram(d=fc_degree, n_additional_pts=fc_cont_points, matrices_path=fcgram_matrices_path).to(device)
elif fc_backend.lower() == 'legendre':
    extension_func = FCLegendre(d=fc_degree, n_additional_pts=fc_cont_points).to(device)
else:
    raise ValueError("fc_backend must be 'legendre' or 'gram'.")

model = FC_FNO(
    in_channels=3,
    out_channels=6,
    Lengths=(Lx, Ly),
    n_modes=n_modes,
    hidden_channels=hidden_channels,
    n_layers=n_layers,
    FC_obj=extension_func,
    positional_embedding=None,
    non_linearity=F.gelu,
    projection_nonlinearity=F.tanh,
).to(device)
state = torch.load(checkpoint_path, map_location=device, weights_only=False)
model.load_state_dict(state['model_state_dict'] if isinstance(state, dict) and 'model_state_dict' in state else state)
model.eval()
print('params:', sum(p.numel() for p in model.parameters()))

In [ ]:
def split_derivatives(derivs):
    if not isinstance(derivs, (list, tuple)) or len(derivs) != 2:
        raise ValueError('Expected FC_FNO to return [dx, dy] derivatives.')
    return derivs[0], derivs[1]


def averaged_rectangle_rule(values):
    _, _, x_res, y_res = values.shape
    number_of_rectangle_cells = x_res * y_res
    rectangle_cell_volume = (Lx * Ly) / number_of_rectangle_cells
    sampled_domain_volume = rectangle_cell_volume * number_of_rectangle_cells
    rectangle_integral = values.flatten(start_dim=1).sum(dim=1) * rectangle_cell_volume
    return rectangle_integral / sampled_domain_volume


def averaged_rectangle_rule_np(values):
    values = np.asarray(values)
    x_res, y_res = values.shape[-2], values.shape[-1]
    number_of_rectangle_cells = x_res * y_res
    rectangle_cell_volume = (Lx * Ly) / number_of_rectangle_cells
    sampled_domain_volume = rectangle_cell_volume * number_of_rectangle_cells
    rectangle_integral = np.sum(values, axis=(-2, -1)) * rectangle_cell_volume
    return rectangle_integral / sampled_domain_volume


def pointwise_fc_residual_from_tensors(p, pred, dx_pred, dy_pred):
    # Returns pointwise (r1_density, r2_density), each (B, 1, X, Y).
    lambda_, mu = lame_from_p(p)
    grad_U_11 = dx_pred[:, 4:5] + w_grad_tensor[:, 0:1]
    grad_U_12 = dy_pred[:, 4:5] + w_grad_tensor[:, 1:2]
    grad_U_21 = dx_pred[:, 5:6] + w_grad_tensor[:, 2:3]
    grad_U_22 = dy_pred[:, 5:6] + w_grad_tensor[:, 3:4]
    eps_11 = grad_U_11
    eps_22 = grad_U_22
    eps_12 = 0.5 * (grad_U_12 + grad_U_21)
    eps_21 = eps_12
    trace_eps = eps_11 + eps_22
    C_eps_11 = 2.0 * mu * eps_11 + lambda_ * trace_eps
    C_eps_22 = 2.0 * mu * eps_22 + lambda_ * trace_eps
    C_eps_12 = 2.0 * mu * eps_12
    C_eps_21 = 2.0 * mu * eps_21
    T_11 = pred[:, 0:1] + z_grad_tensor[:, 0:1] - C_eps_11
    T_12 = pred[:, 1:2] + z_grad_tensor[:, 1:2] - C_eps_12
    T_21 = pred[:, 2:3] + z_grad_tensor[:, 2:3] - C_eps_21
    T_22 = pred[:, 3:4] + z_grad_tensor[:, 3:4] - C_eps_22
    trace_T = T_11 + T_22
    r1 = (T_11.square() + T_12.square() + T_21.square() + T_22.square()) / (2.0 * mu)
    r1 = r1 - (lambda_ / (4.0 * mu * (lambda_ + mu))) * trace_T.square()
    div_sigma_1 = dx_pred[:, 0:1] + dy_pred[:, 1:2]
    div_sigma_2 = dx_pred[:, 2:3] + dy_pred[:, 3:4]
    r2 = div_sigma_1.square() + div_sigma_2.square()
    return r1, r2


def fc_residual_terms(p, pred, derivs):
    dx_pred, dy_pred = split_derivatives(derivs)
    r1, r2 = pointwise_fc_residual_from_tensors(p, pred, dx_pred, dy_pred)
    return averaged_rectangle_rule(r1), averaged_rectangle_rule(r2)

pred_chunks = []
dx_chunks = []
dy_chunks = []
fc_residual_loss_dict = {
    'loss_1': np.zeros(num_check),
    'loss_2': np.zeros(num_check),
    'total_loss': np.zeros(num_check),
    'sqrt_total_loss': np.zeros(num_check),
}

with torch.no_grad():
    for start in tqdm(range(0, num_check, batch_size), desc='Checkpoint inference'):
        end = min(start + batch_size, num_check)
        model_input = model_input_tensor[start:end].to(device)
        p_batch = p_tensor[start:end].to(device)
        pred, derivs = model(model_input, derivs_to_compute=['dx', 'dy'])
        dx_pred, dy_pred = split_derivatives(derivs)
        r1_mse, r2_mse = fc_residual_terms(p_batch, pred, derivs)
        total = r1_mse + r2_mse
        fc_residual_loss_dict['loss_1'][start:end] = r1_mse.cpu().numpy()
        fc_residual_loss_dict['loss_2'][start:end] = r2_mse.cpu().numpy()
        fc_residual_loss_dict['total_loss'][start:end] = total.cpu().numpy()
        fc_residual_loss_dict['sqrt_total_loss'][start:end] = torch.sqrt(total).cpu().numpy()
        pred_chunks.append(pred.detach().cpu())
        dx_chunks.append(dx_pred.detach().cpu())
        dy_chunks.append(dy_pred.detach().cpu())

prediction_tensor = torch.cat(pred_chunks, dim=0)
dx_prediction_tensor = torch.cat(dx_chunks, dim=0)
dy_prediction_tensor = torch.cat(dy_chunks, dim=0)
prediction_xy = prediction_tensor.numpy().astype(np.float64)
np.save(model_test_outputs_path / 'visualized_prediction_vertex_values_xy.npy', prediction_xy)
np.save(model_test_outputs_path / 'visualized_prediction_fc_residual_loss_dict.npy', fc_residual_loss_dict)

print('prediction:', tuple(prediction_tensor.shape), 'dx:', tuple(dx_prediction_tensor.shape), 'dy:', tuple(dy_prediction_tensor.shape))
print(f"mean model FC residual loss 1: {np.mean(fc_residual_loss_dict['loss_1']):.2e} (std: {np.std(fc_residual_loss_dict['loss_1']):.2e})")
print(f"mean model FC residual loss 2: {np.mean(fc_residual_loss_dict['loss_2']):.2e} (std: {np.std(fc_residual_loss_dict['loss_2']):.2e})")
print(f"mean model FC total residual: {np.mean(fc_residual_loss_dict['total_loss']):.2e} (std: {np.std(fc_residual_loss_dict['total_loss']):.2e})")

## FC-PINO residual visualization

The figure uses the original pointwise squared residual scale. Each panel uses a grayscale colorbar with limits set by the 1% and 99% quantiles of that FC-grid residual image. The domain is `[0, 2] x [0, 1]`.

In [ ]:
extent = [mesh_args['lower_left_x'], mesh_args['upper_right_x'], mesh_args['lower_left_y'], mesh_args['upper_right_y']]
num_visualize_residual = num_check


def pointwise_fc_residual_arrays(sample_index):
    with torch.no_grad():
        p_sample = p_tensor[sample_index:sample_index + 1].to(device)
        pred = prediction_tensor[sample_index:sample_index + 1].to(device)
        dx_pred = dx_prediction_tensor[sample_index:sample_index + 1].to(device)
        dy_pred = dy_prediction_tensor[sample_index:sample_index + 1].to(device)
        r1, r2 = pointwise_fc_residual_from_tensors(p_sample, pred, dx_pred, dy_pred)
    return r1.squeeze().detach().cpu().numpy(), r2.squeeze().detach().cpu().numpy()


def fc_quantile_limits(values, lower=0.01, upper=0.99):
    finite_values = np.asarray(values)[np.isfinite(values)]
    if finite_values.size == 0:
        return 0.0, 1.0
    vmin, vmax = np.quantile(finite_values, [lower, upper])
    if not np.isfinite(vmin) or not np.isfinite(vmax):
        return float(np.nanmin(finite_values)), float(np.nanmax(finite_values))
    if np.isclose(vmin, vmax):
        delta = max(abs(float(vmin)) * 1e-6, 1e-12)
        vmin -= delta
        vmax += delta
    return float(vmin), float(vmax)


def plot_fc_pino_residual_pair(fc_r1, fc_r2, local_sample_index):
    global_sample_index = sample_start_index + local_sample_index
    residuals = [
        (fc_r1, r'FC-PINO residual $r_1^2$'),
        (fc_r2, r'FC-PINO residual $r_2^2$'),
    ]

    fig, axs = plt.subplots(2, 1, figsize=(11, 11), constrained_layout=False)
    fig.subplots_adjust(left=0.04, right=0.9, bottom=0.06, top=0.92, hspace=0.18)

    for ax, (values, title_base) in zip(axs, residuals):
        mean_value = float(averaged_rectangle_rule_np(values))
        vmin, vmax = fc_quantile_limits(values)
        im = ax.imshow(values.T, extent=extent, origin='lower', cmap='gray', vmin=vmin, vmax=vmax)
        ax.set_title(f'{title_base}\n(mean={mean_value:.2e})', fontsize=22, pad=12)
        ax.set_aspect('equal', adjustable='box')
        ax.set_xticks([])
        ax.set_yticks([])
        cbar = fig.colorbar(im, ax=ax, fraction=0.025, pad=0.02)
        cbar.locator = ticker.MaxNLocator(nbins=4)
        formatter = ticker.ScalarFormatter(useMathText=False)
        formatter.set_scientific(True)
        formatter.set_powerlimits((0, 0))
        cbar.formatter = formatter
        cbar.update_ticks()
        cbar.ax.tick_params(labelsize=18)
        cbar.ax.yaxis.get_offset_text().set_fontsize(18)

    path = visualization_dir / f'sample_{global_sample_index}_fc_pino_residual_r1_r2.png'
    fig.savefig(path, dpi=300, bbox_inches='tight')
    plt.close(fig)
    return path

pointwise_residual_visualization_files = []
pointwise_residual_visualization_stats = []
for local_sample_index in tqdm(range(num_visualize_residual), desc='FC-PINO residual visualizations'):
    fc_r1, fc_r2 = pointwise_fc_residual_arrays(local_sample_index)

    pointwise_residual_visualization_files.append(str(plot_fc_pino_residual_pair(fc_r1, fc_r2, local_sample_index)))

    pointwise_residual_visualization_stats.append({
        'sample_index': sample_start_index + local_sample_index,
        'fc_r1_rectangle_average': float(averaged_rectangle_rule_np(fc_r1)),
        'fc_r2_rectangle_average': float(averaged_rectangle_rule_np(fc_r2)),
        'fc_total_rectangle_average': float(averaged_rectangle_rule_np(fc_r1 + fc_r2)),
        'fc_r1_max': float(np.max(fc_r1)),
        'fc_r2_max': float(np.max(fc_r2)),
        'fc_total_max': float(np.max(fc_r1 + fc_r2)),
    })

print(f'Saved {len(pointwise_residual_visualization_files)} FC-PINO residual figure(s) to {visualization_dir}')
print(json.dumps(pointwise_residual_visualization_stats, indent=2))

## Save summary

In [ ]:
def summarize_loss_dict(loss_dict):
    return {key + '_mean': float(np.mean(value)) for key, value in loss_dict.items()} | {key + '_std': float(np.std(value)) for key, value in loss_dict.items()}

metrics_summary = {
    'dataset_split': dataset_split,
    'sample_start_index': sample_start_index,
    'num_check': num_check,
    'checkpoint_path': str(checkpoint_path),
    'fc_backend': fc_backend,
    'fc_degree': fc_degree,
    'fc_cont_points': fc_cont_points,
    'model_architecture': {
        'in_channels': 3,
        'out_channels': 6,
        'n_modes': list(n_modes),
        'hidden_channels': hidden_channels,
        'n_layers': n_layers,
        'input_channels': ['p', 'x', 'y'],
        'output_channels': ['sigma_11', 'sigma_12', 'sigma_21', 'sigma_22', 'u_1', 'u_2'],
    },
    'fc_grid_quadrature': 'averaged rectangle rule: sum(pointwise residual density) * rectangle_cell_volume / (rectangle_cell_volume * number_of_rectangle_cells)',
    'model_prediction_fc_residual': summarize_loss_dict(fc_residual_loss_dict),
    'pointwise_residual_visualization_files': pointwise_residual_visualization_files,
    'pointwise_residual_visualization_stats': pointwise_residual_visualization_stats,
}

summary_path = visualization_dir / 'prediction_pointwise_residual_visualization_summary.json'
with open(summary_path, 'w') as f:
    json.dump(metrics_summary, f, indent=2)

print(json.dumps(metrics_summary, indent=2))
print(f'saved summary to {summary_path}')